In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("C://Users//amogh//Downloads//NIRD 20230130 Database_Hackathon.xlsx", sheet_name="Data Table NIRD 20230130")
df.head()

,STATE_FULL,STATE,COUNTY,ADDRESS,CITY,ZIP_CODE,AHA_ID,HOSPITAL_NAME,TOTAL_BEDS,BURN_BEDS,...,BURN_PEDS,ACS_VERIFIED,TC_STATE_DESIGNATED,ADULT_TRAUMA_L1,ADULT_TRAUMA_L2,PEDS_TRAUMA_L1,PEDS_TRAUMA_L2,ABA_VERIFIED,BC_STATE_DESIGNATED,PHONE
0,Alaska,AK,Anchorage,4315 Diplomacy Dr,Anchorage,99508,6940010.0,Alaska Native Medical Center,173,0,...,0,1,1,0,1,0,1,-1,-1,(907) 563-2662
1,Alaska,AK,Anchorage,3200 Providence Dr,Anchorage,99508,6940020.0,Providence Alaska Medical Center/Children's Ho...,401,0,...,0,1,1,0,1,0,1,-1,-1,(907) 562-2211
2,Alabama,AL,Houston,1108 Ross Clark Cir,Dothan,36301,6530373.0,Southeast Alabama Medical Center,387,0,...,0,0,1,0,1,0,0,-1,-1,(334) 793-8111
3,Alabama,AL,Jefferson,619 19th St S,Birmingham,35233,6530304.0,University of Alabama at Birmingham Hospital (...,1157,28,...,0,1,1,1,0,0,0,0,0,(205) 934-3411
4,Alabama,AL,Jefferson,1600 7th Ave South,Birmingham,35233,6530170.0,Children's of Alabama (Children's of Alabama B...,351,6,...,1,0,1,0,0,1,0,0,0,(205) 638-9100


In [3]:
pip install pgeocode

Note: you may need to restart the kernel to use updated packages.


# Proximity of Non-Burn Trauma Centers to Burn Centers

In [4]:
# Geocode ZIP codes
import pgeocode

nomi = pgeocode.Nominatim('us')

df['lat'] = df['ZIP_CODE'].astype(str).apply(lambda z: nomi.query_postal_code(z).latitude)
df['lon'] = df['ZIP_CODE'].astype(str).apply(lambda z: nomi.query_postal_code(z).longitude)
df = df.dropna(subset=['lat', 'lon'])

In [5]:
# Haversine Distance 
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [6]:
# Create your two subsets AFTER geocoding
non_burn_trauma = df[
    ((df['TRAUMA_ADULT'] == 1) | (df['TRAUMA_PEDS'] == 1)) &
    (df['BURN_ADULT'] != 1) &
    (df['BURN_PEDS'] != 1)
].copy()

burn_centers = df[
    (df['BURN_ADULT'] == 1) | (df['BURN_PEDS'] == 1)
].copy()

In [7]:
# Add dummy variables
non_burn_trauma['key'] = 1
burn_centers['key'] = 1


In [8]:
# Merge with suffixes
pairs = non_burn_trauma.merge(
    burn_centers,
    on='key',
    suffixes=('_trauma', '_burn')
)

In [9]:
# Compute distance
pairs['distance_miles'] = haversine(
    pairs['lat_trauma'], pairs['lon_trauma'],
    pairs['lat_burn'], pairs['lon_burn']
)
# Nearest burn center per trauma center
nearest = pairs.loc[pairs.groupby('HOSPITAL_NAME_trauma')['distance_miles'].idxmin()]
#Avg distance between all burn and trauma centers
avg_distance = nearest['distance_miles'].mean()
print(f"Average distance to nearest burn center: {avg_distance:.2f} miles")

Average distance to nearest burn center: 51.03 miles


In [10]:
nearest

,STATE_FULL_trauma,STATE_trauma,COUNTY_trauma,ADDRESS_trauma,CITY_trauma,ZIP_CODE_trauma,AHA_ID_trauma,HOSPITAL_NAME_trauma,TOTAL_BEDS_trauma,BURN_BEDS_trauma,...,ADULT_TRAUMA_L1_burn,ADULT_TRAUMA_L2_burn,PEDS_TRAUMA_L1_burn,PEDS_TRAUMA_L2_burn,ABA_VERIFIED_burn,BC_STATE_DESIGNATED_burn,PHONE_burn,lat_burn,lon_burn,distance_miles
23972,Illinois,IL,Will,500 Remington Blvd,Bolingbrook,60440,6430034.0,AMITA Health Adventist Medical Center Bolingbr...,138,0,...,1,0,0,0,1,0,(708) 216-9000,41.8793,-87.8433,17.765771
19876,Illinois,IL,DuPage,701 Winthrop Ave,Glendale Heights,60139,6431790.0,AMITA Health Adventist Medical Center GlenOaks,143,0,...,1,0,0,0,1,0,(708) 216-9000,41.8793,-87.8433,12.466282
20004,Illinois,IL,DuPage,120 North Oak St,Hinsdale,60521,6431920.0,AMITA Health Adventist Medical Center Hinsdale,276,0,...,1,0,0,0,1,0,(708) 216-9000,41.8793,-87.8433,7.019319
18084,Illinois,IL,Cook,5101 Willow Springs Rd,La Grange,60525,6432055.0,AMITA Health Adventist Medical Center La Grange,196,0,...,1,0,0,0,1,0,(708) 216-9000,41.8793,-87.8433,6.701719
18596,Illinois,IL,Cook,800 West Biesterfield Road,Elk Grove Village,60007,6431613.0,AMITA Health Alexian Brothers Medical Center,376,0,...,1,0,0,0,1,0,(708) 216-9000,41.8793,-87.8433,11.741062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57954,West Virginia,WV,Monongalia,1 Medical Center Dr,Morgantown,26505,6350007.0,West Virginia University Hospitals (J.W. Ruby ...,690,0,...,1,0,0,0,1,-1,(412) 232-8111,40.4423,-79.9830,54.747579
58082,West Virginia,WV,Ohio,1 Medical Park,Wheeling,26003,6350830.0,Wheeling Hospital,223,0,...,1,0,0,0,1,-1,(412) 232-8111,40.4423,-79.9830,42.167231
56599,Virginia,VA,Winchester,1840 Amherst St,Winchester,22601,6341200.0,Winchester Medical Center,495,0,...,1,0,0,0,1,1,(202) 877-7000,38.9327,-77.0322,64.154610
12441,Florida,FL,Duval,800 Prudential Dr,Jacksonville,32207,6390134.0,Wolfson Children's Hospital,216,0,...,1,0,0,0,1,1,(352) 265-0111,29.6813,-82.3539,60.327409


In [11]:
nearest = nearest.rename(
    columns = {'HOSPITAL_NAME_trauma': 'HOSPITAL_NAME',
    'HOSPITAL_NAME_burn': 'Nearest_Burn_Center',
    'distance_miles': 'Distance_to_Burn'
})
df2 = df.copy()
df2 = df2.merge(
    nearest[['HOSPITAL_NAME', 'Nearest_Burn_Center', 'Distance_to_Burn']],
    on = 'HOSPITAL_NAME',
    how = 'left'
)
df2['Nearest_Burn_Center'] = df2['Nearest_Burn_Center'].fillna('N/A')
df2['Distance_to_Burn'] = df2['Distance_to_Burn'].fillna(0)

df2.columns

Index(['STATE_FULL', 'STATE', 'COUNTY', 'ADDRESS', 'CITY', 'ZIP_CODE',
       'AHA_ID', 'HOSPITAL_NAME', 'TOTAL_BEDS', 'BURN_BEDS', 'TRAUMA_ADULT',
       'TRAUMA_PEDS', 'BURN_ADULT', 'BURN_PEDS', 'ACS_VERIFIED',
       'TC_STATE_DESIGNATED', 'ADULT_TRAUMA_L1', 'ADULT_TRAUMA_L2',
       'PEDS_TRAUMA_L1', 'PEDS_TRAUMA_L2', 'ABA_VERIFIED',
       'BC_STATE_DESIGNATED', 'PHONE', 'lat', 'lon', 'Nearest_Burn_Center',
       'Distance_to_Burn'],
      dtype='object')

In [12]:
# Assign tiers
def assign_tier(row):
    if row['ABA_VERIFIED'] == 1 or row['BC_STATE_DESIGNATED'] == 1:
        return 3
    elif row['TRAUMA_ADULT'] == 1 or row['TRAUMA_PEDS'] == 1:
        return 2
    else:
        return 1

df['tier'] = df.apply(assign_tier, axis=1)

In [13]:
# Build pairwise distance table (cross join)

df['key'] = 1
pairs = df.merge(df, on='key', suffixes=('_A', '_B'))

pairs['distance'] = haversine(
    pairs['lat_A'], pairs['lon_A'],
    pairs['lat_B'], pairs['lon_B']
)


In [14]:
# Nearest trauma center for each local hospital (Tier 1 → Tier 2)
pairs_t2 = pairs[pairs['tier_B'] == 2].dropna(subset=['distance'])

idx_local_to_trauma = pairs_t2.groupby('HOSPITAL_NAME_A')['distance'].idxmin()

local_to_trauma = pairs_t2.loc[idx_local_to_trauma]


In [15]:
# Nearest burn center for each trauma center (Tier 2 → Tier 3)
pairs_t3 = pairs[pairs['tier_B'] == 3].dropna(subset=['distance'])

idx_trauma_to_burn = pairs_t3.groupby('HOSPITAL_NAME_A')['distance'].idxmin()

trauma_to_burn = pairs_t3.loc[idx_trauma_to_burn]


# Referral Chain (Local → Trauma → Burn)

In [16]:
# Build full referral chain (Local → Trauma → Burn)
referral_chain = local_to_trauma.merge(
    trauma_to_burn[['HOSPITAL_NAME_A', 'HOSPITAL_NAME_B', 'distance']],
    left_on='HOSPITAL_NAME_B',
    right_on='HOSPITAL_NAME_A',
    suffixes=('_local_to_trauma', '_trauma_to_burn')
)

referral_chain = referral_chain.rename(columns={
    'HOSPITAL_NAME_A_local_to_trauma': 'Local_Hospital',
    'HOSPITAL_NAME_B_local_to_trauma': 'Nearest_Trauma',
    'HOSPITAL_NAME_B_trauma_to_burn': 'Nearest_Burn',
    'distance_local_to_trauma': 'Distance_Local_to_Trauma',
    'distance_trauma_to_burn': 'Distance_Trauma_to_Burn'
})

referral_chain['Total_Referral_Distance'] = (
    referral_chain['Distance_Local_to_Trauma'] +
    referral_chain['Distance_Trauma_to_Burn']
)
referral_chain

,STATE_FULL_A,STATE_A,COUNTY_A,ADDRESS_A,CITY_A,ZIP_CODE_A,AHA_ID_A,Local_Hospital,TOTAL_BEDS_A,BURN_BEDS_A,...,BC_STATE_DESIGNATED_B,PHONE_B,lat_B,lon_B,tier_B,Distance_Local_to_Trauma,HOSPITAL_NAME_A_trauma_to_burn,Nearest_Burn,Distance_Trauma_to_Burn,Total_Referral_Distance
0,Illinois,IL,Will,500 Remington Blvd,Bolingbrook,60440,6430034.0,AMITA Health Adventist Medical Center Bolingbr...,138,0,...,-1,(630) 312-5000,41.6976,-88.0873,2,0.000000,AMITA Health Adventist Medical Center Bolingbr...,Loyola University Medical Center (Loyola Burn ...,17.765771,17.765771
1,Illinois,IL,DuPage,701 Winthrop Ave,Glendale Heights,60139,6431790.0,AMITA Health Adventist Medical Center GlenOaks,143,0,...,-1,(630) 545-8000,41.9205,-88.0793,2,0.000000,AMITA Health Adventist Medical Center GlenOaks,Loyola University Medical Center (Loyola Burn ...,12.466282,12.466282
2,Illinois,IL,DuPage,120 North Oak St,Hinsdale,60521,6431920.0,AMITA Health Adventist Medical Center Hinsdale,276,0,...,-1,(630) 856-9000,41.8001,-87.9287,2,0.000000,AMITA Health Adventist Medical Center Hinsdale,Loyola University Medical Center (Loyola Burn ...,7.019319,7.019319
3,Illinois,IL,Cook,5101 Willow Springs Rd,La Grange,60525,6432055.0,AMITA Health Adventist Medical Center La Grange,196,0,...,-1,(708) 245-9000,41.7842,-87.8689,2,0.000000,AMITA Health Adventist Medical Center La Grange,Loyola University Medical Center (Loyola Burn ...,6.701719,6.701719
4,Illinois,IL,Cook,800 West Biesterfield Road,Elk Grove Village,60007,6431613.0,AMITA Health Alexian Brothers Medical Center,376,0,...,-1,(847) 437-5500,42.0076,-87.9931,2,0.000000,AMITA Health Alexian Brothers Medical Center,Loyola University Medical Center (Loyola Burn ...,11.741062,11.741062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588,New York,NY,Westchester,100 Woods Rd,Valhalla,10595,6215150.0,Westchester Medical Center/Maria Fareri Childr...,624,10,...,-1,(718) 960-9000,40.8486,-73.8999,2,17.574492,St. Barnabas Hospital,NYC Health + Hospitals / Jacobi Medical Center...,3.081909,20.656401
589,West Virginia,WV,Ohio,1 Medical Park,Wheeling,26003,6350830.0,Wheeling Hospital,223,0,...,-1,(304) 243-3000,40.1027,-80.6476,2,0.000000,Wheeling Hospital,UPMC Mercy (UPMC Mercy Burn Center),42.167231,42.167231
590,Virginia,VA,Winchester,1840 Amherst St,Winchester,22601,6341200.0,Winchester Medical Center,495,0,...,-1,(540) 536-8000,39.1858,-78.1827,2,0.000000,Winchester Medical Center,MedStar Washington Hospital Center (The Burn C...,64.154610,64.154610
591,Florida,FL,Duval,800 Prudential Dr,Jacksonville,32207,6390134.0,Wolfson Children's Hospital,216,0,...,-1,(904) 202-8000,30.2908,-81.6321,2,0.000000,Wolfson Children's Hospital,UF Health Shands Hospital (UF Health Shands Bu...,60.327409,60.327409


# Number of Hops to the Nearest Burn Center

In [17]:
# KPI: Hops

df['hops'] = df['tier'].map({1: 2, 2: 1, 3: 0})
avg_hops = df['hops'].mean()

# KPI: % of hospitals requiring >1 transfer
pct_two_hops = (df['hops'] == 2).mean()*100

# KPI: Bottleneck Index7
inbound = local_to_trauma['HOSPITAL_NAME_B'].value_counts()
df['inbound_referrals'] = df['HOSPITAL_NAME'].map(inbound).fillna(0)

df['bottleneck_index'] = df['inbound_referrals'] / (df['BURN_BEDS'] + 1)

print("Average hops:", avg_hops)
print("% needing >1 transfer:", pct_two_hops)
print("Top bottlenecks:")
print(df[['HOSPITAL_NAME','STATE','bottleneck_index']].sort_values('bottleneck_index', ascending=False).head(50))


Average hops: 0.8583473861720068
% needing >1 transfer: 1.1804384485666104
Top bottlenecks:
                                         HOSPITAL_NAME STATE  bottleneck_index
24                                 UAMS Medical Center    AR               3.0
572                     Cook Children's Medical Center    TX               3.0
599                          Henrico Doctors' Hospital    VA               3.0
556           Ben Taub Hospital - Harris Health System    TX               3.0
563                       Covenant Children's Hospital    TX               3.0
557                  HCA Houston Healthcare Clear Lake    TX               3.0
539        University Hospital (Pediatric Burn Center)    TX               3.0
463                       Kettering Health Main Campus    OH               3.0
471                                  OU Medical Center    OK               3.0
508               Einstein Medical Center Philadelphia    PA               3.0
452                               Grant

# No. of Hops - Texas Focus

In [18]:
# Texas Focus
df_tx = df[df['STATE']=='TX'].copy()
print(f"Total No. of Hospitals in Texas: {len(df_tx)}")
print('This includes local hospitals, trauma centers and burn centers.')

# Local -> Trauma Referral
local_to_trauma_tx = local_to_trauma[
    local_to_trauma['STATE_A'] == 'TX'
].copy()

print(f"TX hospitals with a Local → Trauma referral: {len(local_to_trauma_tx)}")
print("These are TX hospitals that are NOT trauma centers and must transfer to one.\n")

# Trauma → Burn referrals
trauma_to_burn_tx = trauma_to_burn[
    trauma_to_burn['STATE_A'] == 'TX'
].copy()

print(f"TX trauma centers with a Trauma → Burn referral: {len(trauma_to_burn_tx)}")
print("These are TX trauma centers that must transfer patients to a burn center.\n")

# Referral chain
referral_chain_tx = local_to_trauma_tx.merge(
    trauma_to_burn_tx[['HOSPITAL_NAME_A', 'HOSPITAL_NAME_B', 'distance']],
    left_on='HOSPITAL_NAME_B',
    right_on='HOSPITAL_NAME_A',
    suffixes=('_local_to_trauma', '_trauma_to_burn')
)

referral_chain_tx = referral_chain_tx.rename(columns={
    'HOSPITAL_NAME_A_local_to_trauma': 'Local_Hospital',
    'HOSPITAL_NAME_B_local_to_trauma': 'Nearest_Trauma',
    'HOSPITAL_NAME_B_trauma_to_burn': 'Nearest_Burn',
    'distance_local_to_trauma': 'Distance_Local_to_Trauma',
    'distance_trauma_to_burn': 'Distance_Trauma_to_Burn'
})

referral_chain_tx['Total_Referral_Distance'] = (
    referral_chain_tx['Distance_Local_to_Trauma'] +
    referral_chain_tx['Distance_Trauma_to_Burn']
)

print(f"TX hospitals with a complete Local → Trauma → Burn chain: {len(referral_chain_tx)}")
print("These hospitals require TWO transfers to reach definitive burn care.\n")

# Compute hops for TX

df_tx['hops'] = df_tx['tier'].map({1: 2, 2: 1, 3: 0})

avg_hops_tx = df_tx['hops'].mean()
pct_two_hops_tx = (df_tx['hops'] == 2).mean() * 100

print("HOPS SUMMARY FOR TEXAS")
print("--------------------------------")
print(f"Average number of hops: {avg_hops_tx:.2f}")
print(f"Percent of TX hospitals requiring >1 transfer (2 hops): {pct_two_hops_tx:.2f}%")
print(f"TX hospitals requiring 2 hops: {(df_tx['hops'] == 2).sum()}")
print(f"TX hospitals requiring 1 hop: {(df_tx['hops'] == 1).sum()}")
print(f"TX hospitals requiring 0 hops: {(df_tx['hops'] == 0).sum()}\n")

print("Sample TX Referral Chains (Local → Trauma → Burn):")
print(referral_chain_tx[['Local_Hospital', 'Nearest_Trauma', 'Nearest_Burn',
                         'Distance_Local_to_Trauma', 'Distance_Trauma_to_Burn',
                         'Total_Referral_Distance']].head())
print("\nAnalysis complete.\n")


Total No. of Hospitals in Texas: 48
This includes local hospitals, trauma centers and burn centers.
TX hospitals with a Local → Trauma referral: 48
These are TX hospitals that are NOT trauma centers and must transfer to one.

TX trauma centers with a Trauma → Burn referral: 48
These are TX trauma centers that must transfer patients to a burn center.

TX hospitals with a complete Local → Trauma → Burn chain: 48
These hospitals require TWO transfers to reach definitive burn care.

HOPS SUMMARY FOR TEXAS
--------------------------------
Average number of hops: 0.90
Percent of TX hospitals requiring >1 transfer (2 hops): 2.08%
TX hospitals requiring 2 hops: 1
TX hospitals requiring 1 hop: 41
TX hospitals requiring 0 hops: 6

Sample TX Referral Chains (Local → Trauma → Burn):
                                    Local_Hospital  \
0                             Ascension Seton Hays   
1                       Ascension Seton Williamson   
2    Baylor Scott & White Hillcrest Medical Center   
3 